# California Housing Price Prediction - Exercise

In this recap exercise you will work through a complete (small) machine learning project: you load the California Housing dataset, explore it visually, create a representative test set, clean and transform the data with Scikit-Learn transformers and pipelines, and finally train and evaluate a first linear regression model.

The data is in `datasets/housing/housing.csv` (it is downloaded to `datasets/housing.tgz` automatically if it is missing). The California map used in the *nice to know* section is `images/california.png`.

Reference values from the solution are given in *italics* so you can check your results.

### Learning Objectives
By the end of this notebook, you will be able to:
- Load and preprocess real-world data
- Handle missing values and categorical features
- Create data visualization for insights
- Build and evaluate machine learning models
- Apply feature scaling and transformation techniques

---
---
---
## Complete the tasks in bold

# 0-Setup

First, let's import a few common modules and ensure MatplotLib plots figures inline. We also check that Python 3.5 or later is installed (although Python 2.x may work, it is deprecated so we strongly recommend you use Python 3 instead), as well as Scikit-Learn ≥ 1.0.1.

**TASK: Run the cell below to import the libraries you will need.**

In [ ]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 7)

# Scikit-Learn ≥ 1.0.1 is required
from packaging import version
import sklearn
print(sklearn.__version__)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

# Common imports
import numpy as np
import pandas as pd
import os

# To plot pretty figures
import matplotlib as mpl
import matplotlib.pyplot as plt


# 1-Data Loading and Exploration

## Dataset Overview
We'll be working with the California Housing dataset, which contains data from the 1990 California census. Our goal is to predict median house values in California districts.

### Features Description
- `longitude`, `latitude`: Geographic location
- `housing_median_age`: Median age of houses in the district
- `total_rooms`: Total number of rooms
- `total_bedrooms`: Total number of bedrooms
- `population`: Number of people in the district
- `households`: Number of households
- `median_income`: Median income in the district
- `ocean_proximity`: Location relative to the ocean
- `median_house_value`: Target variable (what we want to predict)

## Download the Data

**TASK: Write a function `load_housing_data()` that checks whether `datasets/housing.tgz` exists. If not, it creates the `datasets` folder, downloads the file from `https://github.com/ageron/data/raw/main/housing.tgz` (use `urllib.request.urlretrieve`) and extracts it into `datasets` (use `tarfile`). The function returns `pd.read_csv(Path("datasets/housing/housing.csv"))`. Call it and store the result in a dataframe called `housing`.**

## Take a Quick Look at the Data Structure

**TASK: Show the first 5 rows of `housing`.**

**TASK: Show the shape of `housing`. *(Expected: (20640, 10))***

**TASK: Use `info()` to show the data types and the number of non-null values per column.**

**QUESTION: Which column has missing values, and how many? Which column is not numerical? *(Hint: there are 20640 rows.)***

*Your answer:*

**TASK: Show the column names of `housing`.**

## Exploratory Data Analysis

## Goals of This Section
1. Understand the data distribution
2. Identify patterns and relationships
3. Detect potential data quality issues
4. Gain insights for feature engineering

## Key Visualizations
We'll explore:
1. Geographic distribution of house prices
2. Correlation between features
3. Distribution of numerical features
4. Patterns in categorical data

Remember: Good visualization is key to understanding your data and making informed decisions about preprocessing and modeling.

The describe() method shows a summary of
the numerical attributes

**TASK: Use `describe()` to show a summary of the numerical attributes.**

#### Plot a histogram for each numerical attribute

**TASK: Plot a histogram of every numerical attribute with `housing.hist(bins=50, figsize=(12, 8))`. *(Optional: set smaller default font sizes first with `plt.rc(...)`, e.g. `plt.rc('xtick', labelsize=10)`.)***

**QUESTION: Look at the histograms and the `describe()` output. What do you notice about:
1. the units/scale of `median_income`?
2. the maximum values of `housing_median_age` and `median_house_value` (the target)? Why could this be a problem, and what could you do about it?
3. the scales of the different attributes?
4. the shape of the distributions (symmetrical or tail-heavy)?**

*Your answer:*

## Visualizing Geographical Data

**TASK: Make a scatter plot of the districts with `longitude` on the x-axis and `latitude` on the y-axis (`housing.plot(kind="scatter", ...)`).**

Setting the alpha option to 0.1 makes it much easier to visualize the places where there is a high density of data points

**TASK: Repeat the scatter plot, now with `alpha=0.2`.**

**QUESTION: Which areas have a high density of districts?**

*Your answer:*

Now let’s look at the housing prices. The radius of each circle represents the district’s population (option s), and the color represents the
price (option c). We will use a predefined color map (option cmap) called jet, which ranges from blue (low values) to red (high prices). 

The argument `sharex=False` fixes a display bug (the x-axis values and legend were not displayed). This is a temporary fix (see: https://github.com/pandas-dev/pandas/issues/10611 ). Thanks to Wilmer Arellano for pointing it out.

**TASK: Make the scatter plot of longitude/latitude again, with `grid=True`, the marker size `s` equal to `housing["population"] / 100` (`label="population"`), the colour `c` equal to `"median_house_value"` with `cmap="jet"`, `colorbar=True`, `legend=True`, `sharex=False` and `figsize=(10, 7)`.**

**QUESTION: What does this plot tell you about the relation between location and housing prices?**

*Your answer:*

#### START NICE TO KNOW

**TASK: Run the cell below, which sets the path to the California map image.**

In [ ]:
# Use existing California image from images folder
images_path = os.path.join(".", "images")
filename = "california.png"

**TASK: Load the image with `matplotlib.image.imread(os.path.join(images_path, filename))` into `california_img`. Make the scatter plot of the previous task (this time with `alpha=0.4` and `colorbar=False`) and overlay the map with `plt.imshow(california_img, extent=[-124.55, -113.80, 32.45, 42.05], alpha=0.5)`. *(Optional: add a colorbar with tick labels in $k.)***

#### END NICE TO KNOW

# 2-Data Processing

## Overview
In this section, we'll prepare our data for machine learning algorithms through several steps:

1. **Data Cleaning**
   - Handling missing values
   - Converting categorical data to numerical
   
2. **Feature Engineering**
   - Feature scaling
   - Creating transformation pipelines
   
3. **Data Transformation**
   - Standardization
   - One-hot encoding for categorical variables

These steps are crucial for ensuring our data is in the right format and scale for machine learning algorithms to work effectively.

## Create a Test Set
### Why Split the Data?
Before any data exploration or preprocessing, we need to separate our test set to avoid data leakage. This ensures that:
- We have a truly independent test set
- Our model evaluation will be realistic
- We don't accidentally use information from the test set during development

## Splitting Approaches
We'll use two methods:
1. Random splitting (simple but may introduce sampling bias)
2. Stratified sampling (ensures representative splits based on income categories)

### Important Warning ⚠️
Never preprocess the test set! It should remain untouched until the final evaluation.

**TASK: Set the numpy random seed to 42 so that the output of this notebook is identical at every run.**

If you don't use _np.random.seed(42)_ and you run the program again, it will generate a different test set! Over time, you (or your Machine Learning
algorithms) will get to see the whole dataset, which is what you want to avoid.

Scikit-Learn provides a few functions to split datasets into multiple subsets in various ways. The simplest function is _train_test_split()_

**TASK: Use `train_test_split` to split `housing` into `train_set` and `test_set` with `test_size=0.2` and `random_state=42`.**

**TASK: Show the first rows of `test_set`. *(The first index should be 20046.)***

- Random sampling is fine if your dataset is large enough (relative to the number of attributes)
- But you run the risk of introducing a significant sampling bias
- Suppose the median income is a very important attribute to predict median housing prices.
- Then make sure that you have +/- the same distribution in your test set as in the whole dataset. 
- This is called __stratified sampling__ which is especially useful if you a have skew dataset.
- Since the median income is a continuous numerical attribute, you first need to create an income category attribute.



**TASK: Create a new column `housing["income_cat"]` using `pd.cut` on `median_income` with `bins=[0., 1.5, 3.0, 4.5, 6., np.inf]` and `labels=[1, 2, 3, 4, 5]`.**

**TASK: Show the number of districts in each income category. *(Expected: category 3 = 7236, 2 = 6581, 4 = 3639, 5 = 2362, 1 = 822)***

**TASK: Plot a histogram of `income_cat`.**

See https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedShuffleSplit.html

**TASK: Make a stratified split: use `train_test_split` with `test_size=0.2`, `stratify=housing["income_cat"]` and `random_state=42`. Store the result in `strat_train_set` and `strat_test_set`.**

Let's check if this worked: are income category proportions in test set equal to complete dataset? 

**TASK: Calculate the proportion of each income category in `strat_test_set` and in the complete `housing` dataset.**

**QUESTION: Are the income category proportions in the test set (approximately) equal to those of the complete dataset? *(Expected: category 3 is about 0.3505 in both.)***

*Your answer:*

**TASK: Now remove the `income_cat` column from both `strat_train_set` and `strat_test_set`.**

**TASK: Create `housing` as a copy of `strat_train_set` without the `median_house_value` column (the features), and `housing_labels` as a copy of the `median_house_value` column of `strat_train_set` (the labels).**

## Data Cleaning

Most Machine Learning algorithms cannot work with missing features, so let’s create a few functions to take care of them. We saw earlier that the
total_bedrooms attribute has some missing values, so let’s fix this. You have three options:

```python
housing.dropna(subset=["total_bedrooms"])    # option 1
housing.drop("total_bedrooms", axis=1)       # option 2
median = housing["total_bedrooms"].median()  # option 3
housing["total_bedrooms"].fillna(median, inplace=True)
```

To demonstrate each of them, let's create a copy of the housing dataset, but keeping only the rows that contain at least one null. Then it will be easier to visualize exactly what each option does:

**TASK: Create `sample_incomplete_rows`: the first 5 rows of `housing` that contain at least one missing value (*hint: `housing.isnull().any(axis=1)`*). Show it.**

**TASK: Apply option 1 (`dropna` on `total_bedrooms`) to `sample_incomplete_rows`. What remains?**

**TASK: Apply option 2 (drop the `total_bedrooms` column) to `sample_incomplete_rows`.**

**TASK: Apply option 3: calculate the `median` of `housing["total_bedrooms"]` and fill the missing values of `sample_incomplete_rows["total_bedrooms"]` with it (*hint: `sample_incomplete_rows["total_bedrooms"] = sample_incomplete_rows["total_bedrooms"].fillna(median)`*). Show the result. *(Expected median: 434.0)***

We decide to choose option 3 and use the sklearn class SimpleImputer

**TASK: Import `SimpleImputer` and create an `imputer` with `strategy="median"`.**

Remove the text attribute because median can only be calculated on numerical attributes:

**TASK: Create `housing_num` by dropping the `ocean_proximity` column from `housing`. *(Alternative: `housing.select_dtypes(include=[np.number])`)***

**TASK: Fit the `imputer` on `housing_num`.**

Only the total_bedrooms attribute had missing values, but we cannot be sure that there won’t be any missing values in new data after the system goes live, so it is safer to apply the imputer to all the numerical attributes:

**TASK: Show the medians learned by the imputer (`imputer.statistics_`). *(Expected: [-118.51, 34.26, 29., 2125., 434., 1167., 408., 3.5385])***

Check that this is the same as manually computing the median of each attribute:

**TASK: Check that `imputer.statistics_` equals the manually computed median of each column of `housing_num`.**

Now you can use this “trained” imputer to transform the training set by replacing missing values with the learned medians:

**TASK: Transform `housing_num` with the trained imputer and store the result in `X`.**

The result is a plain NumPy array containing the transformed features. If you want to put it back into a pandas DataFrame, it’s simple:

**TASK: Convert `X` back into a dataframe `housing_tr`, with the columns and index of `housing_num`.**

**TASK: Show the first rows and the `describe()` of `housing_tr`. *(Expected: count = 16512 for every column, so no missing values left.)***

## Handling Text and Categorical Attributes

Now let's preprocess the categorical input feature, `ocean_proximity`:

**TASK: Store the `ocean_proximity` column as a dataframe (double brackets!) in `housing_cat` and show the first 10 rows.**

Most Machine Learning algorithms prefer to work with numbers, so let’s convert these categories from text to numbers.

**TASK: Use an `OrdinalEncoder` (`ordinal_encoder`) to `fit_transform` `housing_cat` into `housing_cat_encoded`. Show the first 10 values. *(Expected first values: 3, 0, 1, 1, 4, ...)***

**TASK: Show the categories found by the encoder. *(Expected: ['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'])***

**QUESTION: What is the problem with representing `ocean_proximity` like this when you feed it to an ML algorithm? *(Hint: which categories get numbers that are close to each other, and are they really similar?)***

*Your answer:*

A common **solution is to create one binary attribute per category: one attribute equal to 1 when the category is “<1H OCEAN” (and 0 otherwise), another attribute equal to 1 when the category is “INLAND” (and 0 otherwise)**, and so on. The new attributes are sometimes called dummy attributes.  

This is called **One Hot Encoding**. 

**TASK: Use a `OneHotEncoder` (`cat_encoder`) to `fit_transform` `housing_cat` into `housing_cat_1hot` and show the result. *(Expected: a sparse matrix of shape (16512, 5))***

Notice that the output is a SciPy sparse matrix, instead of a NumPy array. This is very useful when you have categorical attributes with thousands of
categories. After one-hot encoding, we get a matrix with thousands of columns, and the matrix is full of 0s except for a single 1 per row. Using up
tons of memory mostly to store zeros would be very wasteful, so instead a sparse matrix only stores the location of the nonzero elements. You can use it
mostly like a normal 2D array, but if you really want to convert it to a (dense) NumPy array, just call the toarray() method:

**TASK: Convert `housing_cat_1hot` to a dense NumPy array.**

Alternatively, you can set `sparse=False` when creating the `OneHotEncoder`:

**TASK: Create the `OneHotEncoder` again, now with `sparse_output=False`, and `fit_transform` `housing_cat` again.**

**TASK: Show `cat_encoder.categories_`, `cat_encoder.feature_names_in_` and `cat_encoder.get_feature_names_out()`.**

## Feature Scaling

One of the most important transformations you need to apply to your data is feature scaling. With few exceptions, Machine Learning algorithms don’t perform well when the input numerical attributes have very different scales. 

This is the case for the housing data: 
- the total number of rooms ranges from about 6 to 39.320
-  the median incomes only range from 0 to 15. 

Note that scaling the target values is generally not required. There are two common ways to get all attributes to have the same scale: 

- min-max scaling
- standardization.

__Min-max scaling__ (aka __normalization__)
- simplest scaler
- values are shifted and rescaled so that they end up ranging from 0 to 1. 
- subtract the min value and divide by the max minus the min. 
- Scikit-Learn provides a transformer called `MinMaxScaler` for this. 

__Standardization__ 
- first it subtracts the mean value (so standardized values always have a zero mean)
- then it divides by the standard deviation so that the resulting distribution has unit variance. 
- Unlike min-max scaling, standardization does not bound values to a specific range, which may be a problem for some algorithms (e.g., neural networks often expect an input value ranging from 0 to 1). 
- Standardization is much less affected by outliers. 
- For example, suppose a district had a median income equal to 100 (by mistake). Min-max scaling would then crush all the other values from 0–
15 down to 0–0.15, whereas standardization would not be much affected.

__WARNING__

As with all the transformations, it is important to _fit the scalers to the training data only, not to the full dataset_ (including the test set)only: never use `fit()` or `fit_transform()` for anything
else than the training set.. Only then can you use them to transform the training set and the test set (and new data).

**TASK: Use a `MinMaxScaler` with `feature_range=(-1, 1)` (`min_max_scaler`) to scale `housing_num` into `housing_num_min_max_scaled`.**

**TASK: Use a `StandardScaler` (`std_scaler`) to scale `housing_num` into `housing_num_std_scaled`.**

**QUESTION: Check the min/max of `housing_num_min_max_scaled` and the mean/standard deviation of `housing_num_std_scaled` (e.g. with `.min(axis=0)`, `.mean(axis=0)`, ...). Are they what you expect? Which of the two scalers is less affected by outliers, and why?**

*Your answer:*

When a feature’s distribution has a heavy tail (i.e., when values far from the mean
are not exponentially rare), both min-max scaling and standardization will squash
most values into a small range. Machine learning models generally don’t like this
at all, as you will see in Chapter 4. So before you scale the feature, you should first
transform it to shrink the heavy tail, and if possible to make the distribution roughly
symmetrical. For example, a common way to do this for positive features with a heavy
tail to the right is to replace the feature with its square root (or raise the feature to
a power between 0 and 1). If the feature has a really long and heavy tail, then replacing the feature with its logarithm may help. The  figure below shows how much better this feature looks
when you compute its log: it’s very close to a Gaussian distribution (i.e., bell-shaped).

**TASK: Make two histograms next to each other (`plt.subplots(1, 2, figsize=(8, 3), sharey=True)`, `bins=50`): one of `housing["population"]` and one of the log of the population (`np.log`). Label the axes.**

**QUESTION: How does the log transformation change the distribution of `population`?**

*Your answer:*

## Transformation Pipelines
Scikit-Learn provides the `Pipeline` class to help with such sequences of transformations.

- The Pipeline constructor takes a list of name/estimator pairs defining a sequence of steps. 
- All but the last estimator must be transformers (i.e., they must have a fit_transform() method). 
- The names can be anything you like (as long as they are unique and don’t contain double underscores, __); they will come in handy later for hyperparameter tuning. 
- The estimators must all be transformers (i.e., they must have a `fit_transform()` method), except for the last one, which can be anything: a transformer, a predictor, or any other type of estimator.
- When you call the pipeline’s fit() method, it calls fit_transform() sequentially on all transformers, passing the output of each call as the parameter to the next call until it reaches the final estimator, for which it calls the fit() method.
  
- The pipeline exposes the same methods as the final estimator. 
- In this example, the last estimator is a `StandardScaler`, which is a transformer, so the pipeline has a `transform()` method that applies all the transforms to the data in sequence (and of course also a `fit_transform()` method, which is the one we used).
  
Now let's build a pipeline for preprocessing the numerical attributes:

**TASK: Create a `Pipeline` called `num_pipeline` with two steps: `("impute", SimpleImputer(strategy="median"))` and `("standardize", StandardScaler())`.**

- If you don’t want to name the transformers, you can use the `make_pipeline()` function instead.
- It takes transformers as positional arguments and creates a `Pipeline` using the names of the transformers’ classes, in lowercase and without underscores (e.g., "simpleimputer"):

**TASK: Create the same `num_pipeline` with `make_pipeline()`.**

When you call the pipeline’s `fit()` method, it calls `fit_transform()` sequentially on
all the transformers, passing the output of each call as the parameter to the next call
until it reaches the final estimator, for which it just calls the `fit()` method.

**TASK: Apply `num_pipeline` with `fit_transform` to `housing_num` and store the result in `housing_num_prepared`. Show the first two rows, rounded to 2 decimals. *(Expected first row: [-1.42, 1.01, 1.86, 0.31, 1.37, 0.14, 1.39, -0.94])***

- Now also add the categorical attribute(s). 
- It would be more convenient to have a single transformer capable of handling all columns, applying the appropriate transformations to each column. 
- For this, you can use a `ColumnTransformer`. 
- For example, the following `ColumnTransformer` will apply 
  - `num_pipeline` (the one we just defined) to the numerical attributes and 
  - `cat_pipeline` to the categorical attribute:

**TASK: Define the lists `num_attribs` (the 8 numerical column names) and `cat_attribs` (`["ocean_proximity"]`). Create `cat_pipeline` with `make_pipeline` using `SimpleImputer(strategy="most_frequent")` and `OneHotEncoder(handle_unknown="ignore")`. Then create a `ColumnTransformer` called `preprocessing` that applies `num_pipeline` to `num_attribs` (name `"num"`) and `cat_pipeline` to `cat_attribs` (name `"cat"`).**

- Since listing all the column names is not very convenient, Scikit-Learn provides a `make_column_selector()` function that returns a selector function you can use to automatically select all the features of a given type, such as numerical or categorical.
- You can pass this selector function to the `ColumnTransformer` instead of column names or indices. 
- Moreover, if you don’t care about naming the transformers, you can use `make_column_transformer()`, which chooses the names for you, just like `make_pipeline()` does. 
- For example, the following code creates the same `ColumnTransformer` as earlier, except the transformers are automatically named "pipeline-1" and "pipeline-2" instead of "num" and "cat":

**TASK: Create the same `preprocessing` with `make_column_transformer()` and `make_column_selector()`: `num_pipeline` for `dtype_include=np.number` and `cat_pipeline` for `dtype_include=object`.**

Now we’re ready to apply this `ColumnTransformer` to the housing data:

**TASK: Apply `preprocessing` with `fit_transform` to `housing` and store the result in `housing_prepared`. Show it and check its shape.**

**QUESTION: `housing_prepared` has 13 columns, while `housing` has 9. Where do the extra columns come from?**

*Your answer:*

# 3. Model Development

## Model Selection
In this section, we'll:
1. Train a basic linear regression model
2. Evaluate its performance using RMSE
3. Discuss model limitations and potential improvements

### Why Linear Regression?
- Simple to understand and implement
- Serves as a good baseline model
- Helps identify if we need more complex models

### Evaluation Metrics
We'll use Root Mean Square Error (RMSE) to evaluate our model's performance:
- RMSE measures the standard deviation of the predictions from the actual values
- Lower RMSE values indicate better model performance
- The metric is in the same units as our target variable (dollars)

Now we are ready to select and train a machine learning model. Start with a very basis model: linear regression (You know this from data science fundamentals correct?). 

## Training

**TASK: Create a pipeline `lin_reg` with `make_pipeline` that combines `preprocessing` and a `LinearRegression()` model. Train it on `housing` and `housing_labels`.**

Let's try the full preprocessing pipeline on a few training instances:

**TASK: Use `lin_reg` to predict the labels of the training set and store them in `housing_predictions`. *(Expected first predictions: about 270887, 334831, 119856, ...)***

# 4-Evaluation

**TASK: Show the true labels (`housing_labels.values`) so you can compare them with the predictions.**

**QUESTION: Compare the first predictions with the true labels. Does the model predict well for every district?**

*Your answer:*

Remember that we chose to use the RMSE as our performance measure, so we want to measure this regression model’s RMSE on the whole training set. Use Scikit-Learn’s `mean_squared_error()` function and take the square root (`np.sqrt`) of the result.

RMSE on training set: 

**TASK: Calculate the RMSE on the training set and store it in `lin_rmse`. *(Expected: about 68232.8)***

**QUESTION: Is this a good score? *(Hint: most districts have a `median_house_value` between about \$120.000 and \$265.000.)* Is the model underfitting or overfitting the training data? What does that mean, and what are possible ways to fix it?**

*Your answer:*

## Great work!

----